# Customer Churn Prediction System (TensorFlow)

## Proje Amacı
Bir müşterinin şirketten ayrılıp ayrılmayacağını (churn) tahmin eden
basit bir yapay sinir ağı modeli geliştirmek.

## TensorFlow Neyi Çözer?
TensorFlow, çok boyutlu veri yapıları (tensorler) üzerinden
matematiksel işlemler yaparak makine öğrenmesi modelleri kurmamızı sağlar.

## Bu Projede Kapsanan Konular
- Tensor mantığı
- Dense katmanlar
- Aktivasyon fonksiyonları (ReLU, Sigmoid)
- Regresyon vs sınıflandırma farkı
- Train / Validation / Test ayrımı
- Binary classification problemi

In [2]:
### 🟦 Hücre 2 – Veri Oluşturma (NumPy)
# - NumPy ile sentetik müşteri verisi oluştur
# - Feature matrisi (X)
# - Target (y)

#📌 Amaç:
# Tensor → NumPy → TensorFlow akışı,

import numpy as np
import pandas as pd
import tensorflow as tf

np.random.seed(42)

# 1000 müşteri oluştur
n = 1000

age = np.random.randint(18, 70, n)
monthly_spending = np.random.normal(500, 150, n)
subscription_length = np.random.randint(1, 60, n)
support_calls = np.random.randint(0, 10, n)

# Feature matrisi
X = np.column_stack((age, monthly_spending, subscription_length, support_calls))

# Churn mantığı (basit simülasyon)
churn_probability = (
    0.3 * (support_calls / 10) +
    0.4 * (1 - subscription_length / 60) +
    0.3 * (monthly_spending < 400)
)

y = (churn_probability > 0.5).astype(int)

print("Veri hazırlandı.")
print("X shape:", X.shape)
print("y shape:", y.shape)

Veri hazırlandı.
X shape: (1000, 4)
y shape: (1000,)


In [3]:
### 🟦 Hücre 3 – Train / Test Ayırma
#- Veriyi:
    #- %70 train
    #- %15 validation
    #- %15 test ayır

#Amaç:
#Set kavramını anlamak

from sklearn.model_selection import train_test_split

# %70 train
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# %15 validation - %15 test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (700, 4)
Validation: (150, 4)
Test: (150, 4)


In [4]:
### 🟦 Hücre 4 – TensorFlow Modeli Kurma
#- `Sequential` model oluştur
#- En az:
#    - 2 hidden layer
#    - 1 output layer
#📌 `Dense` kullanımı zorunlu

model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(4,)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225 (900.00 B)

 Trainable params: 225 (900.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
### 🟦 Hücre 5 – Aktivasyon Fonksiyonları
#- Hidden layer → `relu`
#- Output layer → `sigmoid`
#📌 Yorum satırıyla:
#- Neden sigmoid?
#- Neden relu?

# ReLU:
# Negatif değerleri 0 yapar, öğrenmeyi hızlandırır.
# Hidden layer'larda karmaşık ilişkileri yakalamayı sağlar.

# Sigmoid:
# Çıktıyı 0-1 aralığına sıkıştırır.
# Bu yüzden binary classification (churn 0/1) için uygundur.

In [5]:
### 🟦 Hücre 6 – Model Derleme (Compile)
#- Loss:
#    - `binary_crossentropy`
#- Optimizer:
#    - `adam`
#- Metric:
#    - `accuracy`
#📌 Amaç:
#Sınıflandırma–regresyon farkı

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',  # Çünkü binary sınıflandırma
    metrics=['accuracy']
)

In [6]:
### 🟦 Hücre 7 – Model Eğitimi
#- `fit()` kullan
#- Epoch ve batch size belirle
#- Validation set kullan

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val)
)

Epoch 1/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.2795 - loss: 66.1026 - val_accuracy: 0.2467 - val_loss: 41.6700
Epoch 2/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2822 - loss: 29.4285 - val_accuracy: 0.4800 - val_loss: 2.3909
Epoch 3/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6959 - loss: 2.2403 - val_accuracy: 0.7533 - val_loss: 2.1518
Epoch 4/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6840 - loss: 1.9774 - val_accuracy: 0.6800 - val_loss: 1.6153
Epoch 5/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6443 - loss: 1.8591 - val_accuracy: 0.6667 - val_loss: 1.5046
Epoch 6/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6880 - loss: 1.3541 - val_accuracy: 0.6467 - val_loss: 1.4195
Epoch 7/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6990 - loss: 1.3129 - val_accuracy: 0.6533 - val_loss: 1.2904
Epoch 8/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6867 - loss: 1.2621 - val_accuracy: 0.7667 - val_l

In [7]:
### 🟦 Hücre 8 – Model Değerlendirme
#- Test set üzerinde evaluate
#- Accuracy yorumla
#📌 Overfitting var mı? (kısa yorum)

test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", round(test_accuracy * 100, 2), "%")

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8420 - loss: 0.3471 
Test Accuracy: 83.33 %


In [ ]:
# Validation accuracy train’den çok düşükse → overfitting olabilir
# Yakınsa → model dengeli

In [8]:
### 🟦 Hücre 9 – Tahmin Yapma
#- Yeni bir müşteri için churn tahmini yap
#- Çıktıyı yorumla (olasılık → karar)

# Yeni müşteri:
# Yaş: 30
# Aylık harcama: 350
# Abonelik süresi: 6 ay
# Destek talebi: 7

new_customer = np.array([[30, 350, 6, 7]])

prediction = model.predict(new_customer)

print("Churn Olasılığı:", prediction[0][0])

if prediction[0][0] > 0.5:
    print("Riskli müşteri (Ayrılabilir)")
else:
    print("Kalma ihtimali yüksek")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
Churn Olasılığı: 0.8463135
Riskli müşteri (Ayrılabilir)


In [ ]:
## Yönetici Özeti

• Model doğruluğu: %84
• Yüksek riskli müşteriler erken tespit edilebilir
• Sadakat kampanyaları risk grubuna yönlendirilebilir
• Proje ML pipeline mantığını göstermektedir

Bu sistem gerçek verilerle ölçeklenebilir.